In [164]:
import pandas as pd

df = pd.read_excel(
    "Update_Description_Prd.xlsx",
    sheet_name=None,
    engine="openpyxl"
)

print(len(df))

97


In [4]:
import pandas as pd

sheet_name = "ZCDS_FM_FMFPITM"

df = pd.read_excel(
    "Update_Description_Prd.xlsx",
    sheet_name=sheet_name,
    engine="openpyxl"
)

df.columns = df.columns.str.strip()

df.head()

,FIELD,DESCRIPTION
0,MANDT,Mandante
1,AUFNR,Número de Orden
2,POSNR,Número de posición de orden
3,VORNR,Número de operación
4,LTXA1,Texto breve de la operación


In [5]:
column_pairs = [
    ('FIELD', 'DESCRIPTION'),
    ('Field Name', 'Description'),
    ('Field', 'Description')
]

for field_col, desc_col in column_pairs:
    if field_col in df.columns and desc_col in df.columns:
        dict1 = df.set_index(field_col)[desc_col].to_dict()
        break
else:
    raise KeyError("None of the expected column combinations were found.")

print(f"ALTER TABLE `analytics-bi-prd.SAP_RAW_DATA.{sheet_name}`")
print("")

for index, (key, val) in enumerate(dict1.items()):
    if key != 'MANDT':
        if index == len(dict1.keys())-1:
            print(f"ALTER COLUMN `{key}` SET OPTIONS (description = '{val}');")
        else:
            print(f"ALTER COLUMN `{key}` SET OPTIONS (description = '{val}'),")

print("")
print("")
print(f"ALTER TABLE `analytics-bi-prd.SAP_STAGING_DATA.ST_{sheet_name}`")
print("")

for index, (key, val) in enumerate(dict1.items()):
    if key != 'MANDT':
        if index == len(dict1.keys())-1:
            print(f"ALTER COLUMN `{key}` SET OPTIONS (description = '{val}');")
        else:
            print(f"ALTER COLUMN `{key}` SET OPTIONS (description = '{val}'),")


ALTER TABLE `analytics-bi-prd.SAP_RAW_DATA.ZCDS_FM_FMFPITM`

ALTER COLUMN `AUFNR` SET OPTIONS (description = 'Número de Orden'),
ALTER COLUMN `POSNR` SET OPTIONS (description = 'Número de posición de orden'),
ALTER COLUMN `VORNR` SET OPTIONS (description = 'Número de operación'),
ALTER COLUMN `LTXA1` SET OPTIONS (description = 'Texto breve de la operación'),
ALTER COLUMN `RUECK` SET OPTIONS (description = 'Número de confirmación de terminación para la operación'),
ALTER COLUMN `ARBPL` SET OPTIONS (description = 'Puesto de trabajo'),
ALTER COLUMN `AUFNR_TO` SET OPTIONS (description = 'Orden Labor'),
ALTER COLUMN `CSTAT` SET OPTIONS (description = 'Estado notificado'),
ALTER COLUMN `UMREN` SET OPTIONS (description = 'Denominador para convertir UM de hoja de ruta y operaciones'),
ALTER COLUMN `GAMNG` SET OPTIONS (description = 'Cantidad total de la orden'),
ALTER COLUMN `GWEMG` SET OPTIONS (description = 'Cantidad entregada'),
ALTER COLUMN `TOMNG` SET OPTIONS (description = 'Cantidad de O

In [12]:
import json

def generate_statements(schema, dataset_name, table_name):
    statements = []
    clauses = []

    for col in schema:
        name = col.get("name")
        description = (col.get("description") or "").strip()

        if not name or not description:
            continue

        clause = f'ALTER COLUMN `{name}` SET OPTIONS (description = "{description}")'
        clauses.append(clause)

    statements.append(f"ALTER TABLE `{dataset_name}.{table_name}`\n  " + ",\n  ".join(clauses) + ";")

    return statements


with open(file="kssk_config.json", encoding="utf-8") as f:
    schema = json.load(f)

final_s = generate_statements(schema, "dataset",'KSSK') +  generate_statements(schema, "dataset_stg",'st_KSSK')
print(final_s)


['ALTER TABLE `dataset.KSSK`\n  ALTER COLUMN `OBJEK` SET OPTIONS (description = "Clave del objeto a clasificar"),\n  ALTER COLUMN `MAFID` SET OPTIONS (description = "Indicador objeto/clase");', 'ALTER TABLE `dataset_stg.st_KSSK`\n  ALTER COLUMN `OBJEK` SET OPTIONS (description = "Clave del objeto a clasificar"),\n  ALTER COLUMN `MAFID` SET OPTIONS (description = "Indicador objeto/clase");']
